# Nationally Determined Parameters (NDP) Demonstration

This notebook demonstrates the NDP system for managing Eurocode parameters that vary by National Annex.

## Features Covered
1. **Basic Lookups** - Getting NDP values
2. **Context Switching** - Changing code version and country
3. **Temporary Overrides** - Using `ndp_override()` context manager
4. **Custom Annexes** - Registering project-specific parameters at runtime

In [ ]:
from materials.reinforced_concrete.ndp import (
    # Enums
    EurocodeVersion,
    CountryCode,
    # Basic lookup
    get_ndp,
    get_ndp_callable,
    get_ndp_info,
    # Context management
    set_ndp_context,
    get_ndp_context,
    # Runtime customization
    ndp_override,
    register_custom_annex,
    unregister_custom_annex,
    list_custom_annexes,
)

## 1. Basic NDP Lookups

The simplest way to get an NDP value is with `get_ndp()`.

In [ ]:
# Check current context
code, country = get_ndp_context()
print(f"Current context: {code} / {country}")

In [ ]:
# Get some common NDP values
gamma_c = get_ndp("gamma_c")
gamma_s = get_ndp("gamma_s")
alpha_cc = get_ndp("alpha_cc")

print(f"gamma_c (concrete partial factor): {gamma_c}")
print(f"gamma_s (steel partial factor): {gamma_s}")
print(f"alpha_cc (concrete strength coefficient): {alpha_cc}")

In [ ]:
# Get full info including description and code reference
info = get_ndp_info("gamma_c")
print(f"Parameter: gamma_c")
print(f"  Value: {info['value']}")
print(f"  Description: {info['description']}")
print(f"  Reference: {info['ref']}")

### Callable NDPs

Some NDPs are formulas (callables) rather than constants. Use `get_ndp_callable()` for a uniform interface.

In [ ]:
# nu_shear is a formula: 0.6 * (1 - f_ck / 250)
nu_shear_fn = get_ndp_callable("nu_shear")

# Calculate for different concrete strengths
for f_ck in [25, 30, 40, 50]:
    nu = nu_shear_fn(f_ck)
    print(f"f_ck = {f_ck} MPa -> nu_shear = {nu:.3f}")

In [ ]:
# get_ndp_callable also works for constants (wraps them)
gamma_c_fn = get_ndp_callable("gamma_c")
print(f"gamma_c via callable: {gamma_c_fn()}")

## 2. Context Switching

Switch between different Eurocode versions and National Annexes.

In [ ]:
# Available options
print("Available Eurocode versions:")
for v in EurocodeVersion:
    print(f"  - {v}")

print("\nAvailable Country codes:")
for c in CountryCode:
    print(f"  - {c}")

In [ ]:
# Compare alpha_cc across different national annexes
print("alpha_cc values by National Annex:\n")

for country in [CountryCode.EU, CountryCode.EU_UK, CountryCode.EU_DE]:
    set_ndp_context(country=country)
    alpha_cc = get_ndp("alpha_cc")
    print(f"  {country}: alpha_cc = {alpha_cc}")

# Reset to EU
set_ndp_context(country=CountryCode.EU)

In [ ]:
# Compare cot_theta limits
print("cot_theta_upper_lim by National Annex:\n")

for country in [CountryCode.EU, CountryCode.EU_UK, CountryCode.EU_DE]:
    set_ndp_context(country=country)
    cot_max = get_ndp("cot_theta_upper_lim")
    
    if callable(cot_max):
        print(f"  {country}: cot_theta_upper_lim = <formula>")
    else:
        print(f"  {country}: cot_theta_upper_lim = {cot_max}")

# Reset to EU
set_ndp_context(country=CountryCode.EU)

In [ ]:
# Switch code version (Buildings vs Bridges)
print("Comparing EN 1992-1-1 (Buildings) vs EN 1992-2 (Bridges):\n")

for code in EurocodeVersion:
    set_ndp_context(code=code, country=CountryCode.EU)
    gamma_c = get_ndp("gamma_c")
    print(f"  {code}: gamma_c = {gamma_c}")

# Reset
set_ndp_context(code=EurocodeVersion.EN1992_1_1_2004, country=CountryCode.EU)

## 3. Temporary Overrides with Context Manager

Use `ndp_override()` to temporarily change NDP values within a scope.

In [ ]:
# Ensure we start with EU defaults
set_ndp_context(country=CountryCode.EU)

print("Before override:")
print(f"  gamma_c = {get_ndp('gamma_c')}")
print(f"  alpha_cc = {get_ndp('alpha_cc')}")

# Temporary override
with ndp_override(gamma_c=1.6, alpha_cc=0.9):
    print("\nInside override:")
    print(f"  gamma_c = {get_ndp('gamma_c')}")
    print(f"  alpha_cc = {get_ndp('alpha_cc')}")

print("\nAfter override (restored):")
print(f"  gamma_c = {get_ndp('gamma_c')}")
print(f"  alpha_cc = {get_ndp('alpha_cc')}")

In [ ]:
# Override with a callable (formula)
print("Overriding nu_shear with a custom formula:\n")

# Original
nu_fn = get_ndp_callable("nu_shear")
print(f"Original nu_shear(f_ck=30): {nu_fn(30):.3f}")

# Custom formula
custom_nu = lambda f_ck: 0.7 * (1 - f_ck / 300)

with ndp_override(nu_shear=custom_nu):
    nu_fn = get_ndp_callable("nu_shear")
    print(f"Custom nu_shear(f_ck=30): {nu_fn(30):.3f}")

# Back to original
nu_fn = get_ndp_callable("nu_shear")
print(f"Restored nu_shear(f_ck=30): {nu_fn(30):.3f}")

In [ ]:
# Nested overrides
print("Nested overrides:\n")

print(f"Level 0: gamma_c = {get_ndp('gamma_c')}")

with ndp_override(gamma_c=1.6):
    print(f"Level 1: gamma_c = {get_ndp('gamma_c')}")
    
    with ndp_override(gamma_c=1.8):
        print(f"Level 2: gamma_c = {get_ndp('gamma_c')}")
    
    print(f"Level 1 (after inner): gamma_c = {get_ndp('gamma_c')}")

print(f"Level 0 (after outer): gamma_c = {get_ndp('gamma_c')}")

## 4. Custom Annex Registration

Register project-specific parameter sets at runtime without modifying code.

In [ ]:
# Check existing custom annexes
print(f"Registered custom annexes: {list_custom_annexes()}")

In [ ]:
# Register a project-specific annex
register_custom_annex("PROJECT_BRIDGE_X", {
    "gamma_c": 1.4,
    "gamma_s": 1.1,
    "alpha_cc": 0.95,
    "nu_shear": lambda f_ck: 0.65 * (1 - f_ck / 250),
})

print(f"Registered custom annexes: {list_custom_annexes()}")

In [ ]:
# Switch to the custom annex
set_ndp_context(country="PROJECT_BRIDGE_X")

print("Using PROJECT_BRIDGE_X annex:\n")
print(f"  gamma_c = {get_ndp('gamma_c')} (custom: 1.4)")
print(f"  gamma_s = {get_ndp('gamma_s')} (custom: 1.1)")
print(f"  alpha_cc = {get_ndp('alpha_cc')} (custom: 0.95)")

# Parameters not overridden inherit from EU base
print(f"  k_strain = {get_ndp('k_strain')} (inherited from EU)")

In [ ]:
# Custom callable works too
nu_fn = get_ndp_callable("nu_shear")
print(f"Custom nu_shear(f_ck=30): {nu_fn(30):.3f}")

# Compare with EU
set_ndp_context(country=CountryCode.EU)
nu_fn_eu = get_ndp_callable("nu_shear")
print(f"EU nu_shear(f_ck=30): {nu_fn_eu(30):.3f}")

In [ ]:
# Register another annex with client-specific requirements
register_custom_annex("CLIENT_CONSERVATIVE", {
    "gamma_c": 1.6,
    "gamma_s": 1.2,
    "cot_theta_upper_lim": 2.0,  # More conservative strut angle
})

print(f"Registered custom annexes: {list_custom_annexes()}")

In [ ]:
# Compare all contexts
print("Comparison of gamma_c across contexts:\n")

contexts = [
    CountryCode.EU,
    CountryCode.EU_UK,
    CountryCode.EU_DE,
    "PROJECT_BRIDGE_X",
    "CLIENT_CONSERVATIVE",
]

for ctx in contexts:
    set_ndp_context(country=ctx)
    gamma_c = get_ndp("gamma_c")
    print(f"  {ctx:25s}: gamma_c = {gamma_c}")

# Reset
set_ndp_context(country=CountryCode.EU)

In [ ]:
# Validation: try to register with invalid parameter name
try:
    register_custom_annex("INVALID_ANNEX", {
        "gamma_c": 1.5,
        "invalid_param": 999,  # This doesn't exist!
    })
except ValueError as e:
    print(f"Validation error (expected): {e}")

In [ ]:
# Skip validation if needed (for experimental parameters)
register_custom_annex("EXPERIMENTAL", {
    "gamma_c": 1.3,
}, validate=False)

print(f"Registered with validate=False: {list_custom_annexes()}")

In [ ]:
# Clean up: unregister custom annexes
for annex in list_custom_annexes().copy():
    removed = unregister_custom_annex(annex)
    print(f"Unregistered '{annex}': {removed}")

print(f"\nRemaining custom annexes: {list_custom_annexes()}")

## 5. Practical Example: Sensitivity Analysis

Using overrides to quickly compare design outcomes with different parameters.

In [ ]:
def calculate_f_cd(f_ck: float) -> float:
    """Calculate design concrete strength using current NDP context."""
    alpha_cc = get_ndp("alpha_cc")
    gamma_c = get_ndp("gamma_c")
    return alpha_cc * f_ck / gamma_c

# Base case (EU)
set_ndp_context(country=CountryCode.EU)
f_ck = 30  # MPa

print(f"f_ck = {f_ck} MPa\n")
print(f"EU context: f_cd = {calculate_f_cd(f_ck):.2f} MPa")

# UK annex
set_ndp_context(country=CountryCode.EU_UK)
print(f"UK context: f_cd = {calculate_f_cd(f_ck):.2f} MPa")

# Sensitivity: what if gamma_c was higher?
set_ndp_context(country=CountryCode.EU)
with ndp_override(gamma_c=1.6):
    print(f"EU with gamma_c=1.6: f_cd = {calculate_f_cd(f_ck):.2f} MPa")

# Sensitivity: what if alpha_cc was lower?
with ndp_override(alpha_cc=0.8):
    print(f"EU with alpha_cc=0.8: f_cd = {calculate_f_cd(f_ck):.2f} MPa")

## Summary

| Feature | Function | Use Case |
|---------|----------|----------|
| Basic lookup | `get_ndp(param)` | Get single NDP value |
| Callable interface | `get_ndp_callable(param)` | Uniform interface for constants and formulas |
| Full info | `get_ndp_info(param)` | Get value + description + code reference |
| Switch context | `set_ndp_context(code, country)` | Change active code/country globally |
| Check context | `get_ndp_context()` | Get current (code, country) |
| Temp override | `ndp_override(**params)` | Temporarily change values in a scope |
| Custom annex | `register_custom_annex(name, overrides)` | Define project-specific parameters |
| Remove annex | `unregister_custom_annex(name)` | Clean up custom annex |
| List annexes | `list_custom_annexes()` | See registered custom annexes |